# Simplification of Crowell and Labuda for Special Cases using SymPy

## Crowell and Labuda Diffusion MTF Model

The most general analytical diffusion MTF model, from the original Crowell & Labuda (1969) paper "The Silicon Diode Array Camera Tube" (BSTJ Vol. 48, No. 5, Eq. 5).

**Geometry** (Fig. 3 in the paper): Light enters at $y = 0$ (illuminated surface), passes through the undepleted field-free region ($0 \le y \le L_a$), and reaches the depletion region edge at $y = L_a$. The total n-type region extends to $y = L_b$, so the depletion width is $L_b - L_a$.

**Boundary conditions** (Eq. 2):
- At $y = 0$ (illuminated surface): $S p = D \frac{\partial p}{\partial y}$ (surface recombination)
- At $y = L_a$ (depletion edge): $p = 0$ (carriers swept away by E-field)

**Generation** (Eq. 3): $G(x, y) = \frac{N_0}{2} \alpha (1 - R)(1 + \cos kx) e^{-\alpha y}$

where $R$ is the back-surface reflectivity and $k = 2\pi f$ is the spatial frequency.

The collection efficiency at spatial frequency $k$ is given by **Eq. (5)**:

$$\eta_k = \frac{\alpha L (1 - R)}{\alpha^2 L^2 - 1} \left[ \frac{2(\alpha L + SL/D) - (\beta_+ - \beta_-) \exp(-\alpha L_a)}{\beta_+ + \beta_-} - (\alpha L)^{-1} \exp(-\alpha L_a) \right] - (1 - R) \exp(-\alpha L_b)$$

where:
- $\beta_{\pm} = (1 \pm SL/D) \exp(\pm L_a / L)$
- $1/L^2(k) = 1/L_o^2 + k^2$ (so $L(k) = L_o / \sqrt{1 + (2\pi f L_o)^2}$)
- $L_o = \sqrt{D \tau}$ is the diffusion length
- $L_a$ = undepleted (field-free) region thickness
- $L_b$ = total n-type region thickness ($L_b - L_a$ = depletion width)
- $S$ = surface recombination velocity at the illuminated surface
- $D$ = minority carrier diffusion coefficient
- $R$ = back-surface reflectivity

The diffusion MTF is: $\mathrm{MTF}(k) = \eta_k / \eta_0$

## Basic C&L Equation in SymPy

In [ ]:
import sympy as sp

# Define symbols
alpha = sp.Symbol('alpha', positive=True)   # absorption coefficient [1/length]
k     = sp.Symbol('k', nonnegative=True)    # spatial frequency [rad/length]
L_o   = sp.Symbol('L_o', positive=True)    # zero-frequency diffusion length
L_a   = sp.Symbol('L_a', positive=True)    # field-free (undepleted) region thickness
L_b   = sp.Symbol('L_b', positive=True)    # total substrate thickness (L_b > L_a)
S     = sp.Symbol('S', nonnegative=True)    # surface recombination velocity
D     = sp.Symbol('D', positive=True)       # minority carrier diffusion coefficient
R     = sp.Symbol('R', nonnegative=True)    # back-surface reflectivity (0 <= R <= 1)

# Frequency-dependent diffusion length  L(k) = L_o / sqrt(1 + k^2 * L_o^2)
L = L_o / sp.sqrt(1 + k**2 * L_o**2)

# Dimensionless groups
aL  = alpha * L          # alpha * L(k)
sLD = S * L / D          # S*L/D  (dimensionless surface recombination term)

# beta_+/-  (Eq. 5 of Crowell & Labuda 1969)
beta_p = (1 + sLD) * sp.exp( L_a / L)
beta_m = (1 - sLD) * sp.exp(-L_a / L)

# Bracket numerator and denominator
bracket_num = 2*(aL + sLD) - (beta_p - beta_m)*sp.exp(-alpha*L_a)
bracket_den = beta_p + beta_m

# Full C&L collection efficiency eta_k  (Eq. 5)
eta_k = (
    alpha * L * (1 - R) / (alpha**2 * L**2 - 1)
    * (bracket_num / bracket_den - sp.exp(-alpha*L_a) / (alpha*L))
    - (1 - R) * sp.exp(-alpha * L_b)
)

print("eta_k defined. Symbols:", eta_k.free_symbols)

### Inspect sub-expressions

Check $\beta_+$, $\beta_-$ and the bracket before looking at $\eta_k$ in full.

In [ ]:
sp.init_printing(use_unicode=True)

print("β₊ ="); display(beta_p)
print("β₋ ="); display(beta_m)
print("β₊ + β₋ ="); display(sp.simplify(beta_p + beta_m))
print("β₊ - β₋ ="); display(sp.simplify(beta_p - beta_m))

Note that $\beta_+ + \beta_-$ and $\beta_+ - \beta_-$ can be rewritten using hyperbolic functions:

$$\beta_+ + \beta_- = 2\cosh\!\left(\frac{L_a}{L}\right) + 2\frac{SL}{D}\sinh\!\left(\frac{L_a}{L}\right)$$

$$\beta_+ - \beta_- = 2\sinh\!\left(\frac{L_a}{L}\right) + 2\frac{SL}{D}\cosh\!\left(\frac{L_a}{L}\right)$$

This is a cleaner form that we can verify below.

In [ ]:
# Rewrite beta sums using hyperbolic functions and verify they are equivalent
xi = L_a / L   # shorthand for L_a / L(k)

beta_sum_hyp  = 2*sp.cosh(xi) + 2*sLD*sp.sinh(xi)
beta_diff_hyp = 2*sp.sinh(xi) + 2*sLD*sp.cosh(xi)

# Verify identities (should be zero)
check_sum  = sp.simplify((beta_p + beta_m).rewrite(sp.exp) - beta_sum_hyp.rewrite(sp.exp))
check_diff = sp.simplify((beta_p - beta_m).rewrite(sp.exp) - beta_diff_hyp.rewrite(sp.exp))
print("β₊ + β₋  check (should be 0):", check_sum)
print("β₊ - β₋  check (should be 0):", check_diff)

### Full $\eta_k$ in hyperbolic form

Substituting the hyperbolic identities, the bracket ratio becomes:

$$\frac{\beta_+ - \beta_-}{\beta_+ + \beta_-} = \frac{\sinh(L_a/L) + (SL/D)\cosh(L_a/L)}{\cosh(L_a/L) + (SL/D)\sinh(L_a/L)}$$

and the full collection efficiency is:

$$\eta_k = \frac{\alpha L (1-R)}{\alpha^2 L^2 - 1} \left[ \frac{2(\alpha L + SL/D) - \left[2\sinh\!\frac{L_a}{L} + 2\frac{SL}{D}\cosh\!\frac{L_a}{L}\right]e^{-\alpha L_a}}{2\cosh\!\frac{L_a}{L} + 2\frac{SL}{D}\sinh\!\frac{L_a}{L}} - \frac{e^{-\alpha L_a}}{\alpha L} \right] - (1-R)\,e^{-\alpha L_b}$$

In [ ]:
# Rewrite eta_k using hyperbolic form and verify equivalence
eta_k_hyp = (
    alpha * L * (1 - R) / (alpha**2 * L**2 - 1)
    * (
        (2*(aL + sLD) - beta_diff_hyp * sp.exp(-alpha*L_a)) / beta_sum_hyp
        - sp.exp(-alpha*L_a) / (alpha*L)
    )
    - (1 - R) * sp.exp(-alpha * L_b)
)

# Verify equivalence with the original exponential form
check_eta = sp.simplify(eta_k.rewrite(sp.exp) - eta_k_hyp.rewrite(sp.exp))
print("eta_k equivalence check (should be 0):", check_eta)

## Simplifications

### Case 1: Dead Illuminated Surface + No Back-Surface Reflection

**Physical meaning of the two assumptions:**

| Assumption | Parameter change | Physical meaning |
|---|---|---|
| Dead illuminated surface | $S \to \infty$ | Every minority carrier reaching $y=0$ recombines instantly. The surface is a perfect sink, so the minority carrier density vanishes there: $p(0)=0$. |
| No back-surface reflection | $R = 0$ | Photons that reach the back of the substrate ($y = L_b$) are fully absorbed or transmitted — none are reflected back into the field-free region. |

**Effect on $\beta_\pm$ as $S \to \infty$:** The $SL/D \to \infty$ term dominates $\pm 1$ inside $\beta_\pm$, so using the hyperbolic forms:

$$\beta_+ + \beta_- = 2\cosh\!\frac{L_a}{L} + 2\frac{SL}{D}\sinh\!\frac{L_a}{L} \;\xrightarrow{S\to\infty}\; 2\frac{SL}{D}\sinh\!\frac{L_a}{L}$$

$$\beta_+ - \beta_- = 2\sinh\!\frac{L_a}{L} + 2\frac{SL}{D}\cosh\!\frac{L_a}{L} \;\xrightarrow{S\to\infty}\; 2\frac{SL}{D}\cosh\!\frac{L_a}{L}$$

The $SL/D$ factors cancel in the ratio, leaving $\coth(L_a/L)$.

In the numerator, $2(\alpha L + SL/D) \xrightarrow{S\to\infty} 2\,SL/D$, which when divided by $2\,(SL/D)\sinh(L_a/L)$ gives $1/\sinh(L_a/L)$.

The simplified collection efficiency (verified by SymPy below) is:

$$\boxed{\eta_k^{S\to\infty} = \frac{\alpha L}{\alpha^2 L^2 - 1} \left[ \frac{1}{\sinh(L_a/L)} - \coth\!\frac{L_a}{L}\cdot e^{-\alpha L_a} - \frac{e^{-\alpha L_a}}{\alpha L} \right] - e^{-\alpha L_b}}$$

where $L = L_o/\sqrt{1 + k^2 L_o^2}$ as before.

In [ ]:
# Apply S -> inf and R = 0 to eta_k_hyp (hyperbolic form is cleaner for this limit)
eta_k_SE = sp.limit(eta_k_hyp.subs(R, 0), S, sp.oo)
eta_k_SE = sp.simplify(eta_k_SE)
print("eta_k (S->inf, R=0):")
display(eta_k_SE)

# Verify against the closed-form in the text above
eta_k_SE_expected = (
    alpha * L / (alpha**2 * L**2 - 1)
    * (1/sp.sinh(xi) - sp.coth(xi)*sp.exp(-alpha*L_a) - sp.exp(-alpha*L_a)/(alpha*L))
    - sp.exp(-alpha*L_b)
)
check = sp.simplify(eta_k_SE.rewrite(sp.exp) - eta_k_SE_expected.rewrite(sp.exp))
print("\nVerification vs boxed formula (should be 0):", check)

### Comparison with the Stanford / El Gamal formula

The Stanford/El Gamal model also assumes $S \to \infty$ and $R = 0$, but uses a **reversed geometry**: light enters through the **depletion region first** ($0 \le z \le L_D$), then the field-free region ($L_D \le z \le L_D + L$). The dead surface sits at the *far* end of the field-free region, not at the illuminated surface.

$$D(f) = \frac{1 + \alpha L_f - e^{-\alpha L_D}}{1 + \alpha L_f} - \frac{\alpha L_f\, e^{-\alpha L_D}\bigl(e^{-\alpha L} - e^{-L/L_f}\bigr)}{(1 - \alpha^2 L_f^2)\,\sinh(L/L_f)}$$

**These two models are NOT equivalent.** Despite sharing the same simplifying assumptions ($S\to\infty$, $R=0$), the reversed geometry changes both the unnormalized collection efficiency *and* the MTF ratio. We confirm this numerically below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Numerical parameters
alpha_v = 0.05   # absorption coefficient [1/um]
L_o_v   = 10.0   # diffusion length [um]
L_a_v   = 8.0    # field-free region [um]  (= Stanford L)
L_D_v   = 5.0    # depletion width [um]    (= Stanford L_D)
L_b_v   = L_a_v + L_D_v

f_cy_um = np.linspace(0, 0.1, 300)
k_v     = 2 * np.pi * f_cy_um
Lf_v    = L_o_v / np.sqrt(1 + (k_v * L_o_v)**2)
xi_v    = L_a_v / Lf_v
xi_0    = L_a_v / L_o_v

# C&L simplified (S->inf, R=0): dead illuminated surface, light enters field-free first
def eta_CL(Lf, xi):
    aL = alpha_v * Lf
    return (
        aL / (aL**2 - 1)
        * (1/np.sinh(xi) - np.cosh(xi)/np.sinh(xi)*np.exp(-alpha_v*L_a_v)
           - np.exp(-alpha_v*L_a_v)/aL)
        - np.exp(-alpha_v*L_b_v)
    )

MTF_CL = eta_CL(Lf_v, xi_v) / eta_CL(L_o_v, xi_0)

# Stanford/El Gamal: dead far surface, light enters depletion first
def D_stanford(Lf):
    aL  = alpha_v * Lf
    eLd = np.exp(-alpha_v * L_D_v)
    t1  = (1 + aL - eLd) / (1 + aL)
    num = aL * eLd * (np.exp(-alpha_v*L_a_v) - np.exp(-L_a_v/Lf))
    den = (1 - aL**2) * np.sinh(L_a_v/Lf)
    with np.errstate(divide='ignore', invalid='ignore'):
        t2 = np.where(np.abs(den) > 1e-30, num/den, 0.0)
    return t1 - t2

MTF_S = D_stanford(Lf_v) / D_stanford(L_o_v)

print(f"Max MTF difference: {np.max(np.abs(MTF_CL - MTF_S)):.4f}")
print("Conclusion: the two models are NOT equivalent (different geometry)")

plt.figure(figsize=(7, 4))
plt.plot(f_cy_um * 1000, MTF_CL, label="C&L simplified (dead illuminated surface)")
plt.plot(f_cy_um * 1000, MTF_S,  label="Stanford/El Gamal (dead far surface)", linestyle='--')
plt.xlabel("Spatial frequency [cy/mm]")
plt.ylabel("MTF")
plt.title("C&L simplified vs Stanford/El Gamal")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()